In [7]:
import pandas as pd

In [9]:
df = pd.read_csv("data/Biomedical_dataset.csv")
test_df = pd.read_csv("data/Private_dataset.csv")

In [11]:
import itertools

combinations = [list(itertools.combinations(df.columns[:-1], r)) for r in range(1, len(df.columns[:-1]) + 1)]

all_combinations = [list(combo) for sublist in combinations for combo in sublist]

print(all_combinations)


[['mean_distance_question'], ['mean_distance_answer'], ['QRS'], ['ACRS'], ['mean_distance_question', 'mean_distance_answer'], ['mean_distance_question', 'QRS'], ['mean_distance_question', 'ACRS'], ['mean_distance_answer', 'QRS'], ['mean_distance_answer', 'ACRS'], ['QRS', 'ACRS'], ['mean_distance_question', 'mean_distance_answer', 'QRS'], ['mean_distance_question', 'mean_distance_answer', 'ACRS'], ['mean_distance_question', 'QRS', 'ACRS'], ['mean_distance_answer', 'QRS', 'ACRS'], ['mean_distance_question', 'mean_distance_answer', 'QRS', 'ACRS']]


In [15]:
import torch.nn as nn
import tqdm
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
np.random.seed(42)
tf.random.set_seed(42)

metrics = {"Model": [], "Inputs": [],"Train_Accuracy":[], "Test_1_Accuracy": [], "Test_2_Accuracy": [],  "Train_F1":[], "Test_1_F1":[],  
           "Test_2_F1":[], "Train_Precision":[],"Test_1_Precision":[], "Test_2_Precision":[],
           "Train_Sensitivity":[], "Test_1_Sensitivity":[], "Test_2_Sensitivity":[],  
           "Train_AUROC":[], "Test_1_AUROC":[], "Test_2_AUROC":[]}
model_count = 0
for i in all_combinations:
    for j in range(10):
        model_count+=1
        df_this = df[i]
        test_this = test_df[i]
        X_train = df_this
        y_train = df['correctness'].values.astype(np.int32)
        X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)  
        X_val, X_opt, y_val, y_opt = train_test_split(X_val, y_val, test_size=0.5, random_state=42)  
        
        X_test = test_this
        y_test = test_df['correctness'].values.astype(np.int32)  
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_opt = scaler.transform(X_opt)
        X_test = scaler.transform(X_test)
        
        try:
            model = Sequential([
                Dense(4, input_shape=(len(i),), activation='relu'),
                Dense(4, activation='relu'),
                Dense(4, activation='relu'),
                Dense(1, activation='sigmoid') 
            ])
            
            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            model.fit(X_train, y_train, epochs=100, batch_size=16, validation_data=(X_opt, y_opt), verbose=0)
            loss, accuracy = model.evaluate(X_test, y_test)
    
            y_pred_test_2 = (model.predict(X_test) > 0.5).astype("int32")
            y_pred_test_1 = (model.predict(X_val) > 0.5).astype("int32")
            y_pred_opt = (model.predict(X_opt) > 0.5).astype("int32")
            test_acc_score2 = accuracy_score(y_test, y_pred_test_2)
            test_acc_score1 = accuracy_score(y_val, y_pred_test_1)
            test_prec_score2 = precision_score(y_test, y_pred_test_2)
            test_prec_score1 = precision_score(y_val, y_pred_test_1)
            test_f1_score2 = f1_score(y_test, y_pred_test_2)
            test_f1_score1 = f1_score(y_val, y_pred_test_1)
            test_sens_score2 = recall_score(y_test, y_pred_test_2)
            test_sens_score1 = recall_score(y_val, y_pred_test_1)
            test_roc_score2 = roc_auc_score(y_test, y_pred_test_2)
            test_roc_score1 = roc_auc_score(y_val, y_pred_test_1)
    
            train_acc_score = accuracy_score(y_opt, y_pred_opt)
            train_prec_score = precision_score(y_opt, y_pred_opt)
            train_f1_score = f1_score(y_opt, y_pred_opt)
            train_sens_score = recall_score(y_opt, y_pred_opt)
            train_roc_score = roc_auc_score(y_opt, y_pred_opt)
    
            metrics['Model'].append(model_count)
            metrics['Inputs'].append(i)
            
            metrics['Test_2_Accuracy'].append(test_acc_score2)
            metrics['Test_1_Accuracy'].append(test_acc_score1)
            metrics['Test_2_Precision'].append(test_prec_score2)
            metrics['Test_1_Precision'].append(test_prec_score1)
            metrics['Test_2_F1'].append(test_f1_score2)
            metrics['Test_1_F1'].append(test_f1_score1)
            metrics['Test_2_AUROC'].append(test_roc_score2)
            metrics['Test_1_AUROC'].append(test_roc_score1)
            metrics['Test_2_Sensitivity'].append(test_sens_score2)
            metrics['Test_1_Sensitivity'].append(test_sens_score1)
            
            metrics['Train_Accuracy'].append(train_acc_score)
            metrics['Train_Precision'].append(train_prec_score)
            metrics['Train_F1'].append(train_f1_score)
            metrics['Train_AUROC'].append(train_roc_score)
            metrics['Train_Sensitivity'].append(train_sens_score)
    
    
            model.save(f'models/model_{model_count}.h5')
            print("Model saved as " + f'models/model_{model_count}.h5')
        except:
            continue

Couldn't import dot_parser, loading of dot files will not be possible.


/opt/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6771 - loss: 0.6686
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model saved as models/model_1.h5


...

Model saved as models/model_149.h5


/opt/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6937 - loss: 0.6603
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model saved as models/model_150.h5


In [45]:
pd.DataFrame(metrics).to_csv("output/metrics_all_models.csv")

In [21]:
metrics= pd.DataFrame(metrics)

In [23]:
metrics

,Model,Inputs,Train_Accuracy,Test_1_Accuracy,Test_2_Accuracy,Train_F1,Test_1_F1,Test_2_F1,Train_Precision,Test_1_Precision,Test_2_Precision,Train_Sensitivity,Test_1_Sensitivity,Test_2_Sensitivity,Train_AUROC,Test_1_AUROC,Test_2_AUROC
0,1,[mean_distance_question],0.739130,0.772727,0.625,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.500000,0.500000,0.500000
1,2,[mean_distance_question],0.826087,0.727273,0.750,0.600000,0.500000,0.722222,0.750000,0.428571,0.619048,0.500000,0.6,0.866667,0.720588,0.682353,0.773333
2,3,[mean_distance_question],0.826087,0.772727,0.675,0.600000,0.444444,0.580645,0.750000,0.500000,0.562500,0.500000,0.4,0.600000,0.720588,0.641176,0.660000
3,4,[mean_distance_question],0.826087,0.772727,0.750,0.600000,0.545455,0.722222,0.750000,0.500000,0.619048,0.500000,0.6,0.866667,0.720588,0.711765,0.773333
4,5,[mean_distance_question],0.869565,0.727273,0.700,0.666667,0.500000,0.647059,1.000000,0.428571,0.578947,0.500000,0.6,0.733333,0.750000,0.682353,0.706667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,146,"[mean_distance_question, mean_distance_answer,...",0.782609,0.590909,0.475,0.666667,0.400000,0.588235,0.555556,0.300000,0.416667,0.833333,0.6,1.000000,0.799020,0.594118,0.580000
145,147,"[mean_distance_question, mean_distance_answer,...",0.826087,0.818182,0.700,0.600000,0.600000,0.600000,0.750000,0.600000,0.600000,0.500000,0.6,0.600000,0.720588,0.741176,0.680000
146,148,"[mean_distance_question, mean_distance_answer,...",0.913043,0.636364,0.375,0.833333,0.428571,0.509804,0.833333,0.333333,0.361111,0.833333,0.6,0.866667,0.887255,0.623529,0.473333
147,149,"[mean_distance_question, mean_distance_answer,...",0.739130,0.772727,0.625,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.500000,0.500000,0.500000


In [25]:
import pandas as pd

best_rows = []

for start in range(0, len(metrics), 10):
    subset = metrics.iloc[start:start+10] 

    max_test1_accuracy = subset.iloc[:, 3].max()
    
    top_test1_rows = subset[subset.iloc[:, 3] == max_test1_accuracy]
    
    if len(top_test1_rows) > 1:
        
        best_row = top_test1_rows.loc[top_test1_rows.iloc[:, 4].idxmax()]
    else:
        
        best_row = top_test1_rows.iloc[0]

    best_rows.append(best_row)

result_df = pd.DataFrame(best_rows).reset_index(drop=True)



In [27]:
result_df

,Model,Inputs,Train_Accuracy,Test_1_Accuracy,Test_2_Accuracy,Train_F1,Test_1_F1,Test_2_F1,Train_Precision,Test_1_Precision,Test_2_Precision,Train_Sensitivity,Test_1_Sensitivity,Test_2_Sensitivity,Train_AUROC,Test_1_AUROC,Test_2_AUROC
0,4,[mean_distance_question],0.826087,0.772727,0.750,0.600000,0.545455,0.722222,0.750000,0.500000,0.619048,0.500000,0.6,0.866667,0.720588,0.711765,0.773333
1,17,[mean_distance_answer],0.826087,0.818182,0.725,0.600000,0.500000,0.592593,0.750000,0.666667,0.666667,0.500000,0.4,0.533333,0.720588,0.670588,0.686667
2,22,[QRS],0.826087,0.772727,0.775,0.600000,0.444444,0.640000,0.750000,0.500000,0.800000,0.500000,0.4,0.533333,0.720588,0.641176,0.726667
3,32,[ACRS],0.826087,0.818182,0.650,0.600000,0.500000,0.416667,0.750000,0.666667,0.555556,0.500000,0.4,0.333333,0.720588,0.670588,0.586667
4,46,"[mean_distance_question, mean_distance_answer]",0.782609,0.772727,0.725,0.444444,0.545455,0.666667,0.666667,0.500000,0.611111,0.333333,0.6,0.733333,0.637255,0.711765,0.726667
5,57,"[mean_distance_question, QRS]",0.782609,0.772727,0.775,0.444444,0.444444,0.640000,0.666667,0.500000,0.800000,0.333333,0.4,0.533333,0.637255,0.641176,0.726667
6,66,"[mean_distance_question, ACRS]",0.782609,0.818182,0.675,0.444444,0.600000,0.580645,0.666667,0.600000,0.562500,0.333333,0.6,0.600000,0.637255,0.741176,0.660000
7,79,"[mean_distance_answer, QRS]",0.826087,0.818182,0.725,0.600000,0.333333,0.620690,0.750000,1.000000,0.642857,0.500000,0.2,0.600000,0.720588,0.600000,0.700000
8,89,"[mean_distance_answer, ACRS]",0.826087,0.863636,0.700,0.600000,0.666667,0.538462,0.750000,0.750000,0.636364,0.500000,0.6,0.466667,0.720588,0.770588,0.653333
9,94,"[QRS, ACRS]",0.782609,0.772727,0.700,0.444444,0.444444,0.571429,0.666667,0.500000,0.615385,0.333333,0.4,0.533333,0.637255,0.641176,0.666667


In [31]:
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

hi_dict_biomedical = {}
hi_dict_private = {}
for c in range(len(result_df)):
    model_loaded = load_model(f'models/model_{result_df.iloc[c,0]}.h5')
    i = result_df.iloc[c,1]
    df_this = df[i]
    test_this = test_df[i]
    
    X_train = df_this
    y_train = df['correctness'].values.astype(np.int32)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)  
    X_val, X_opt, y_val, y_opt = train_test_split(X_val, y_val, test_size=0.5, random_state=42)  
    
    X_test = test_this
    y_test = test_df['correctness'].values.astype(np.int32)  
    
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_opt = scaler.transform(X_opt)
    X_test = scaler.transform(X_test)
    
    loss, accuracy = model_loaded.evaluate(X_opt, y_opt)
    print(f"Val accuracy of loaded model: {accuracy:.2f}")
    
    loss, accuracy = model_loaded.evaluate(X_val, y_val)
    print(f"Test1 accuracy of loaded model: {accuracy:.2f}")
    
    loss, accuracy = model_loaded.evaluate(X_test, y_test)
    print(f"Test2 accuracy of loaded model: {accuracy:.2f}")
    
    l1 = list(model_loaded.predict(df_this))
    l2 = list(model_loaded.predict(test_this))

    y_pred = (model_loaded.predict(X_test) > 0.5).astype("int32")
    
    hi_dict_biomedical['Biomedical_'+str(result_df.iloc[c,0])] = [float(a[0]) for a in l1]
    hi_dict_private['Private_'+str(result_df.iloc[c,0])] = [float(a[0]) for a in l2]
    


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.8261 - loss: 0.5933
Val accuracy of loaded model: 0.83
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7727 - loss: 0.6064
Test1 accuracy of loaded model: 0.77
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7292 - loss: 0.6281
Test2 accuracy of loaded model: 0.75
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 702us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 717us/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.8261 - loss: 0.6408
Val accuracy of loaded model: 0.83
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8182 - loss: 0.6507
Test1 accuracy of loaded model: 0.82
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7333 - loss: 0.6759
Test2 accuracy of loaded model: 0.73
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 936us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 774us/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.8261 - loss: 0.6468
Val accuracy of loaded model: 0.83
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7727 - loss: 0.6338
Test1 accuracy of loaded model: 0.77
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7771 - loss: 0.6402
Test2 accuracy of loaded model: 0.77
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 691us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 807us/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.8261 - loss: 0.6347
Val accuracy of loaded model: 0.83
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8182 - loss: 0.6447
Test1 accuracy of loaded model: 0.82
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6729 - loss: 0.6602
Test2 accuracy of loaded model: 0.65
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 708us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 785us/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.7826 - loss: 0.5358
Val accuracy of loaded model: 0.78
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7727 - loss: 0.5760
Test1 accuracy of loaded model: 0.77
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7125 - loss: 0.6252
Test2 accuracy of loaded model: 0.73
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 704us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 617us/step


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.7826 - loss: 0.6251
Val accuracy of loaded model: 0.78
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8636 - loss: 0.6160
Test1 accuracy of loaded model: 0.86
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6792 - loss: 0.6433
Test2 accuracy of loaded model: 0.68
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 633us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 686us/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.8696 - loss: 0.6347
Val accuracy of loaded model: 0.87
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8182 - loss: 0.6378
Test1 accuracy of loaded model: 0.82
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6792 - loss: 0.6816
Test2 accuracy of loaded model: 0.68
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 637us/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.8261 - loss: 0.5577
Val accuracy of loaded model: 0.83
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8182 - loss: 0.5754
Test1 accuracy of loaded model: 0.82
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6854 - loss: 0.6130
Test2 accuracy of loaded model: 0.70
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 854us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 903us/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.8261 - loss: 0.6387
Val accuracy of loaded model: 0.83
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7727 - loss: 0.6407
Test1 accuracy of loaded model: 0.77
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6687 - loss: 0.6563
Test2 accuracy of loaded model: 0.68
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 728us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 649us/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.8261 - loss: 0.6208
Val accuracy of loaded model: 0.83
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8182 - loss: 0.6256
Test1 accuracy of loaded model: 0.82
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6958 - loss: 0.6412
Test2 accuracy of loaded model: 0.70
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 844us/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 674us/step


In [33]:
pd.DataFrame(hi_dict_biomedical)

,Biomedical_4,Biomedical_17,Biomedical_22,Biomedical_32,Biomedical_46,Biomedical_57,Biomedical_66,Biomedical_79,Biomedical_89,Biomedical_94,Biomedical_103,Biomedical_114,Biomedical_122,Biomedical_132,Biomedical_147
0,0.517748,0.488930,0.439702,0.447088,0.499526,0.433763,0.450024,0.468926,0.428595,0.414298,0.424594,0.519224,0.384828,0.496893,0.442026
1,0.517748,0.502441,0.439702,0.447088,0.513308,0.434008,0.468155,0.485643,0.467708,0.430458,0.443985,0.543067,0.384828,0.387302,0.453784
2,0.517748,0.495640,0.439702,0.447088,0.513308,0.434463,0.468188,0.479253,0.448243,0.413314,0.436651,0.554737,0.384828,0.391133,0.445540
3,0.451884,0.484699,0.439702,0.447088,0.443377,0.435596,0.434719,0.468693,0.428595,0.423127,0.430352,0.479918,0.384828,0.491661,0.442026
4,0.457352,0.482584,0.439702,0.447088,0.421305,0.435619,0.434719,0.465040,0.428595,0.419152,0.427116,0.477319,0.384828,0.472744,0.442026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218,0.517748,0.497171,0.439702,0.447088,0.513308,0.433732,0.456216,0.482983,0.456109,0.428259,0.436450,0.533768,0.384828,0.392708,0.442774
219,0.512528,0.491249,0.439702,0.447088,0.500235,0.433767,0.444911,0.477789,0.428595,0.424080,0.432108,0.511394,0.384828,0.475456,0.442026
220,0.517748,0.492983,0.439702,0.447088,0.506851,0.433599,0.449246,0.480472,0.434052,0.424824,0.432670,0.519369,0.384828,0.445829,0.442026
221,0.517748,0.489921,0.439702,0.447088,0.501806,0.433742,0.450299,0.471698,0.428595,0.416268,0.426209,0.520425,0.384828,0.484168,0.442026


In [35]:
pd.DataFrame(hi_dict_private)

,Private_4,Private_17,Private_22,Private_32,Private_46,Private_57,Private_66,Private_79,Private_89,Private_94,Private_103,Private_114,Private_122,Private_132,Private_147
0,0.517748,0.494559,0.439702,0.447088,0.513308,0.433735,0.453467,0.482211,0.433696,0.420749,0.431387,0.537773,0.384828,0.438077,0.442026
1,0.517748,0.493343,0.439702,0.447088,0.513308,0.433941,0.456364,0.479902,0.429412,0.412130,0.427728,0.543237,0.384828,0.444917,0.442026
2,0.517748,0.497464,0.439702,0.447088,0.513308,0.435366,0.470743,0.480235,0.451838,0.411023,0.440584,0.566846,0.384828,0.380280,0.448032
3,0.517748,0.512033,0.439702,0.452639,0.513308,0.434018,0.469544,0.497679,0.484876,0.467558,0.459635,0.501422,0.384828,0.392551,0.459848
4,0.517748,0.503680,0.439702,0.447088,0.513308,0.434210,0.475437,0.486045,0.468950,0.428527,0.447229,0.555818,0.384828,0.383859,0.456468
5,0.517748,0.494491,0.439702,0.447088,0.513308,0.433735,0.453467,0.482160,0.433544,0.420749,0.431298,0.537625,0.384828,0.438908,0.442026
6,0.517748,0.501833,0.439702,0.447088,0.513308,0.434086,0.470002,0.485450,0.465259,0.427075,0.442489,0.555165,0.384828,0.382799,0.451454
7,0.517748,0.495505,0.439702,0.447088,0.513308,0.433616,0.451081,0.484011,0.437619,0.427934,0.435209,0.533356,0.384828,0.433097,0.442026
8,0.517748,0.490799,0.439702,0.447088,0.507853,0.433776,0.449394,0.476049,0.428595,0.412130,0.425823,0.532477,0.384828,0.492800,0.442026
9,0.517748,0.495640,0.439702,0.447088,0.513308,0.433735,0.454878,0.483033,0.438988,0.423413,0.432804,0.539741,0.384828,0.421954,0.442026


In [37]:
HI_Biomedical = pd.DataFrame(hi_dict_biomedical)
HI_private = pd.DataFrame(hi_dict_private)

In [39]:
HI_Biomedical_normalized = (HI_Biomedical - HI_Biomedical.min()) / (HI_Biomedical.max() - HI_Biomedical.min())
HI_private_normalized = (HI_private - HI_private.min()) / (HI_private.max() - HI_private.min())


In [47]:
HI_Biomedical_normalized.to_csv("output/HI_Biomedical_NN_normalized_specific.csv")
HI_private_normalized.to_csv("output/HI_private_NN_normalized_specific.csv")